## Streamlined methods to prepare LMF Strata and Segments updated to use the new SDE design tracking schema

Author: Kaitlin Lubetkin, Alex Traynor
Created: 8/12/22  
Last Edited: 09/06/24  

## 1) Initiate variables
<code style="background:yellow;color:black">***(edit for specific analysis)***</code>

In [10]:
import os

#Parent/root folder for analysis
parentFolder = (r'C:\Users\alaurencetraynor\Documents\Tools')

# Analysis name
analysis_name = "example"

# Analysis Area of interest
aoi = "Benchmark Groups"

# SDD geodatabase - this should be a copy you'll use for your analysis
sddsde = os.path.join(parentFolder, ("SDDSDE_"+ analysis_name +".gdb"))

## 2) Dissolve Thiessen Polygons by stratum
<code style="background:yellow;color:black">***(run straight through, no edits needed)***</code>

In [ ]:
# Dissolve Thiessen Polygons by stratum 
arcpy.management.Dissolve(in_features=sddsde + "\\LMFThiessenPolygons",
                          out_feature_class=sddsde + "\\LMFStrata",
                          dissolve_field="Stratum")
arcpy.management.CalculateField(in_table=sddsde + "\\LMFStrata",
                                field="Stratum",
                                expression='!Stratum! + "_LMF"',
                                expression_type="PYTHON3")
# Symbolize to differentiate strata
map = arcpy.mp.ArcGISProject("CURRENT").listMaps("Map")[0]
for lyr in map.listLayers():
    if lyr.name == "LMFStrata":
        sym = lyr.symbology
        sym.updateRenderer("UniqueValueRenderer")
        sym.renderer.fields = "Stratum"
        lyr.symbology = sym

## 3) Check for sampled LMF segments with portions in different strata
<code style="background:yellow;color:black">***(run straight through, no edits needed)***</code>

In [ ]:
# if not already present, add LMF segments and sampled LMF points
map = arcpy.mp.ArcGISProject("CURRENT").listMaps("Map")[0]
if "LMFSegmentPolygons" not in map.listLayers():
    map.addDataFromPath(sddsde + "\\LMFSegmentPolygons")
if "LMF" not in map.listLayers():
    map.addDataFromPath(sddsde + "\\LMF")
    
# update map object with new layers and resymbolize LMFSegmentPolygonsto be hollow
map = arcpy.mp.ArcGISProject("CURRENT").listMaps("Map")[0]
for lyr in map.listLayers():
    if lyr.name == "LMFSegmentPolygons":
        sym = lyr.symbology
        sym.updateRenderer("SimpleRenderer")
        sym.renderer.symbol.applySymbolFromGallery("Black Outline (2 pts)")
        lyr.symbology = sym
        
# select sampled segments intersecting both LMF Type I and Type II
arcpy.management.SelectLayerByAttribute(in_layer_or_view="LMFStrata",
                                        selection_type="NEW_SELECTION", 
                                        where_clause="Stratum = 'Type I_LMF'")
arcpy.management.SelectLayerByLocation(in_layer="LMFSegmentPolygons", 
                                       overlap_type="INTERSECT", 
                                       select_features="LMFStrata", 
                                       selection_type="NEW_SELECTION")
arcpy.management.SelectLayerByAttribute(in_layer_or_view="LMFStrata",
                                        selection_type="NEW_SELECTION", 
                                        where_clause="Stratum = 'Type II_LMF'")
arcpy.management.SelectLayerByLocation(in_layer="LMFSegmentPolygons", 
                                       overlap_type="INTERSECT", 
                                       select_features="LMFStrata", 
                                       selection_type="SUBSET_SELECTION")
arcpy.management.SelectLayerByLocation(in_layer="LMFSegmentPolygons", 
                                       overlap_type="CONTAINS", 
                                       select_features="LMF", 
                                       selection_type="SUBSET_SELECTION")
arcpy.management.SelectLayerByAttribute(in_layer_or_view="LMFStrata",
                                        selection_type="CLEAR_SELECTION")
result = arcpy.management.GetCount("LMFSegmentPolygons")
if int(result[0]) > 0:
    print("Check segments that were selected because they cross strata.")
    print("If all points are in one stratum, clip segment to that stratum.")
    print("If there are points in different strata, split segment, give each a unique SegmentPolygonID, and update the LMF points to reflect new unique SegmentPolygonIDs.")

## 4) Clip segments to AOI
<code style="background:yellow;color:black">***(run straight through, no edits needed)***</code>

In [ ]:
arcpy.management.AddField(sddsde + "\\LMFSegmentPolygons", "area_ratio", "DOUBLE")

arcpy.analysis.Clip(sddsde + "\\LMFSegmentPolygons",
                    aoi,
                    sddsde + "\\LMFSegmentPolygons")

arcpy.management.CalculateField("LMFSegmentPolygons" + thisSeason[1], 
                                    "area_ratio", 
                                    "!Shape_Area!/0.000065", "PYTHON3")

# Clean Arc Pro workspace
map = arcpy.mp.ArcGISProject("CURRENT").listMaps("Map")[0]

# Remove the newly created LMF strata and segment layers from the map
for lyr in map.listLayers():
    #print(lyr.name)
    if lyr.name in ["LMF", "LMFStrata", "LMFStrata_unburnt", "LMFStrata_all", "LMFSegmentPolygons"] + ["LMFSegmentPolygons"]:
        map.removeLayer(lyr)

print("Done!")